# ME302 Course Project: Turbomachine Testing Facility — Scale Analysis

**Authors:** Divyansh Yadav (230387), Hanny (230433), Rohan Kumar (230866)

**Objective:** Find the maximum geometric scale (0 < scale ≤ 1) for testing a turbomachine component in a closed-loop high-pressure facility, while matching the target Mach number (M = 0.5) and Reynolds number (Re = 3 × 10⁶).

## 1. Given Data and Constants

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

# --- Air properties ---
gamma = 1.4                    # Ratio of specific heats
R_gas = 287.0                  # Specific gas constant [J/(kg·K)]
mu = 1.83e-5                   # Dynamic viscosity [Pa·s]

# --- Facility parameters ---
D_pipe = 0.6                   # Pipe diameter [m]
L_diffuser = 1.5               # Available diffuser length [m]
T01 = 293.0                    # Inlet stagnation temperature [K]
p_min_allowed = 200e3          # Minimum allowable pressure [Pa]
l_prototype = 0.2              # Prototype length parameter [m]

# --- Target test conditions ---
M_target = 0.5                 # Target Mach number at test section
Re_target = 3.0e6              # Target Reynolds number

# --- Reference conditions for corrected mass flow ---
p_ref = 101325.0               # Reference pressure [Pa]
T_ref = 288.15                 # Reference temperature [K]

# --- Pressure loss fractions ---
frac_loss_23 = 0.015           # Δp₀₂₃ = 1.5% of p₀₂
frac_loss_45 = 0.05            # Δp₀₄₅ = 5% of p₀₄

print("All constants defined successfully.")

## 2. Load Compressor Operating Map

In [ ]:
data = np.loadtxt('Compressor_operation_map.txt', skiprows=1)
mdot_ref_data = data[:, 0]     # Reference mass flow rate [kg/s]
T0_ratio_data = data[:, 1]     # T02/T01
p0_ratio_data = data[:, 2]     # p02/p01

print(f"Loaded {len(mdot_ref_data)} compressor operating points")
print(f"  mdot_ref range : {mdot_ref_data.min():.4f} – {mdot_ref_data.max():.4f} kg/s")
print(f"  T₀ ratio range : {T0_ratio_data.min():.6f} – {T0_ratio_data.max():.6f}")
print(f"  p₀ ratio range : {p0_ratio_data.min():.6f} – {p0_ratio_data.max():.6f}")

## 3. Isentropic Flow Relations and Diffuser Loss Coefficient

Standard isentropic relations for an ideal gas:

$$\frac{T}{T_0} = \frac{1}{1 + \frac{\gamma - 1}{2} M^2}, \qquad
\frac{p}{p_0} = \left(\frac{T}{T_0}\right)^{\gamma/(\gamma-1)}, \qquad
\frac{\rho}{\rho_0} = \left(\frac{T}{T_0}\right)^{1/(\gamma-1)}$$

Diffuser loss coefficient:
$$K = \begin{cases} 0 & \text{if } \theta < 0 \text{ (nozzle)} \\ 3.2\,\tan^{1.5}\theta\left(1 - \left(\frac{A_3}{A_4}\right)^2\right) & \text{if } \theta > 0 \end{cases}$$

In [ ]:
def isentropic_T_ratio(M):
    """T/T0 for given Mach number."""
    return 1.0 / (1.0 + (gamma - 1.0) / 2.0 * M**2)

def isentropic_p_ratio(M):
    """p/p0 for given Mach number."""
    return isentropic_T_ratio(M) ** (gamma / (gamma - 1.0))

def isentropic_rho_ratio(M):
    """rho/rho0 for given Mach number."""
    return isentropic_T_ratio(M) ** (1.0 / (gamma - 1.0))

def diffuser_loss_coeff(D_in, D_out, L):
    """
    Total pressure loss coefficient K for a conical diffuser.
    K = 0 for semi-angle θ < 0 (nozzle)
    K = 3.2 tan^1.5(θ) (1 - (A_in/A_out)²) for θ > 0
    """
    theta = np.arctan((D_out / 2.0 - D_in / 2.0) / L)
    if theta <= 0:
        return 0.0
    A_ratio = (D_in / D_out) ** 2
    return 3.2 * np.tan(theta)**1.5 * (1.0 - A_ratio**2)

# Quick check
print(f"At M = {M_target}:")
print(f"  T/T₀  = {isentropic_T_ratio(M_target):.6f}")
print(f"  p/p₀  = {isentropic_p_ratio(M_target):.6f}")
print(f"  ρ/ρ₀  = {isentropic_rho_ratio(M_target):.6f}")

## 4. Station-by-Station Analysis

**Flow path:** 1 → Compressor → 2 → Piping (1.5% loss) → 3 → Diffuser/Nozzle → 4 (test section, M = 0.5) → Downstream (5% loss) → 5 → Throttle → 1

**Key relations:**
- $T_{02} = (T_0\text{ ratio}) \times T_{01}$, and $T_{03} = T_{04} = T_{05} = T_{02}$ (adiabatic)
- $p_{02} = (p_0\text{ ratio}) \times p_{01}$
- $p_{03} = 0.985 \, p_{02}$
- $p_{04} = p_{03}$ (since $D_4 < D_3$ for scale $< 1$, the duct is a converging nozzle, $K = 0$)
- $p_{05} = 0.95 \, p_{04}$
- Reynolds number: $Re = \rho_4 V_4 l_{\mathrm{model}} / \mu$
- Corrected mass flow: $\dot{m} = \dot{m}_{\mathrm{ref}} \times (p_{01}/p_{\mathrm{ref}}) / \sqrt{T_{01}/T_{\mathrm{ref}}}$


In [ ]:
def compute_station_conditions(scale, mdot_ref, T0_ratio, p0_ratio):
    """
    For a given scale and compressor operating point, compute all station
    conditions and return the mass-flow balance residual & constraint satisfaction.
    """
    # Geometry
    l_model = scale * l_prototype
    D4 = 2.0 * l_model
    A4 = np.pi / 4.0 * D4**2

    # Stagnation temperature (constant beyond station 2)
    T02 = T0_ratio * T01
    T04 = T02

    # Isentropic ratios at M = 0.5
    T4 = T04 * isentropic_T_ratio(M_target)
    p4_over_p04 = isentropic_p_ratio(M_target)

    # Velocity at test section
    a4 = np.sqrt(gamma * R_gas * T4)
    V4 = M_target * a4

    # From Re constraint
    rho4 = Re_target * mu / (V4 * l_model)
    p4   = rho4 * R_gas * T4
    p04  = p4 / p4_over_p04

    # Diffuser / nozzle loss
    K = diffuser_loss_coeff(D_pipe, D4, L_diffuser)
    p03 = p04   # K = 0 since D4 < D_pipe for scale < 1

    # Pressure chain
    p02 = p03 / (1.0 - frac_loss_23)
    p01_calc = p02 / p0_ratio
    p05 = p04 * (1.0 - frac_loss_45)

    # Mass flow — continuity at test section
    mdot_continuity = rho4 * V4 * A4

    # Mass flow — corrected compressor relation
    mdot_compressor = mdot_ref * (p01_calc / p_ref) / np.sqrt(T01 / T_ref)

    residual = mdot_continuity - mdot_compressor

    return {
        'scale': scale, 'l_model': l_model, 'D4': D4, 'A4': A4,
        'T04': T04, 'T4': T4, 'V4': V4, 'rho4': rho4,
        'p4': p4, 'p04': p04, 'p03': p03, 'p02': p02,
        'p01': p01_calc, 'p05': p05, 'K': K,
        'mdot_continuity': mdot_continuity,
        'mdot_compressor': mdot_compressor,
        'residual': residual,
        'closed_loop_ok': p05 >= p01_calc,
        'min_pressure_ok': (p4 >= p_min_allowed) and (p01_calc >= p_min_allowed),
        'all_ok': (p05 >= p01_calc) and (p4 >= p_min_allowed)
                  and (p01_calc >= p_min_allowed) and (0 < scale <= 1),
    }

print("Analysis function defined.")

## 5. Find Scale for Each Operating Point (Root Finding)

In [ ]:
def find_scale_for_operating_point(idx):
    """Find the scale that satisfies mass-flow balance for compressor point idx."""
    mr = mdot_ref_data[idx]
    tr = T0_ratio_data[idx]
    pr = p0_ratio_data[idx]

    def residual_func(s):
        return compute_station_conditions(s, mr, tr, pr)['residual']

    try:
        r_lo = residual_func(0.01)
        r_hi = residual_func(1.0)
        if r_lo * r_hi > 0:
            return None
        s_sol = brentq(residual_func, 0.01, 1.0, xtol=1e-10)
        return compute_station_conditions(s_sol, mr, tr, pr)
    except Exception:
        return None

# Solve for every compressor point
results = []
for i in range(len(mdot_ref_data)):
    r = find_scale_for_operating_point(i)
    if r is not None:
        results.append((i, r))

valid_results = [(i, r) for i, r in results if r['all_ok']]

print(f"Operating points analysed : {len(mdot_ref_data)}")
print(f"With mass-flow balance    : {len(results)}")
print(f"Satisfying ALL constraints: {len(valid_results)}")

## 6. Optimal Solution — Maximum Scale

In [ ]:
best_idx, best = max(valid_results, key=lambda x: x[1]['scale'])

print("=" * 70)
print("OPTIMAL RESULT")
print("=" * 70)
print(f"Maximum Scale             = {best['scale']:.6f}")
print(f"l_model                   = {best['l_model']*1000:.4f} mm")
print(f"D4 (test section dia)     = {best['D4']*1000:.4f} mm")
print()
print(f"Compressor Operating Point (index {best_idx}):")
print(f"  ṁ_ref                   = {mdot_ref_data[best_idx]:.6f} kg/s")
print(f"  T₀ ratio (T02/T01)      = {T0_ratio_data[best_idx]:.6f}")
print(f"  p₀ ratio (p02/p01)      = {p0_ratio_data[best_idx]:.6f}")
print()
print(f"Inlet stagnation pressure = {best['p01']/1000:.4f} kPa")
print()
print("Station stagnation pressures:")
print(f"  p01 = {best['p01']/1000:.4f} kPa")
print(f"  p02 = {best['p02']/1000:.4f} kPa")
print(f"  p03 = {best['p03']/1000:.4f} kPa")
print(f"  p04 = {best['p04']/1000:.4f} kPa")
print(f"  p05 = {best['p05']/1000:.4f} kPa")
print()
print("Test section (station 4):")
print(f"  p4 (static)  = {best['p4']/1000:.4f} kPa")
print(f"  T4 (static)  = {best['T4']:.4f} K")
print(f"  V4           = {best['V4']:.4f} m/s")
print(f"  ρ4           = {best['rho4']:.6f} kg/m³")
print(f"  ṁ            = {best['mdot_continuity']:.6f} kg/s")
print()
print("Constraint checks:")
print(f"  p05 ≥ p01?  ✓  ({best['p05']/1000:.2f} ≥ {best['p01']/1000:.2f} kPa)")
print(f"  p4  ≥ 200?  ✓  ({best['p4']/1000:.2f} kPa)")
print(f"  p01 ≥ 200?  ✓  ({best['p01']/1000:.2f} kPa)")

## 7. Verification

In [ ]:
Re_check = best['rho4'] * best['V4'] * best['l_model'] / mu
a4_chk   = np.sqrt(gamma * R_gas * best['T4'])
M_check  = best['V4'] / a4_chk

print("--- Verification ---")
print(f"Re (recomputed)  = {Re_check:.1f}  (target {Re_target:.0f})")
print(f"M4 (recomputed)  = {M_check:.6f}  (target {M_target})")
print(f"Mass flow error  = {abs(best['residual']):.2e} kg/s")
print()
print("Pressure chain (forward):")
p01 = best['p01']
p02 = p01 * p0_ratio_data[best_idx]
p03 = p02 * (1 - frac_loss_23)
p04 = p03       # K = 0
p05 = p04 * (1 - frac_loss_45)
print(f"  p01 = {p01/1000:.4f} kPa")
print(f"  p02 = {p02/1000:.4f} kPa  (p01 × {p0_ratio_data[best_idx]:.6f})")
print(f"  p03 = {p03/1000:.4f} kPa  (0.985 × p02)")
print(f"  p04 = {p04/1000:.4f} kPa  (= p03, K = 0)")
print(f"  p05 = {p05/1000:.4f} kPa  (0.95  × p04)")
print(f"  p05 − p01 = {(p05 - p01)/1000:.4f} kPa  (≥ 0  ✓)")

## 8. Compressor Map with Operating Point

The selected operating point (providing the maximum scale) is marked on the $p_0$ ratio vs $\dot{m}_{\mathrm{ref}}$ curve.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- (a) Compressor map ---
ax1 = axes[0]
ax1.scatter(mdot_ref_data, p0_ratio_data, c='steelblue', s=18, alpha=0.7,
            label='Compressor data', zorder=2)
ax1.scatter(mdot_ref_data[best_idx], p0_ratio_data[best_idx],
            c='red', s=200, marker='*', zorder=5,
            label=f'Selected (scale = {best["scale"]:.4f})',
            edgecolors='darkred', linewidths=0.5)
ax1.annotate(
    f'  $\dot{{m}}_{{ref}}$ = {mdot_ref_data[best_idx]:.3f} kg/s\n'
    f'  $p_0$ ratio = {p0_ratio_data[best_idx]:.4f}',
    xy=(mdot_ref_data[best_idx], p0_ratio_data[best_idx]),
    fontsize=9, color='red',
    xytext=(20, 15), textcoords='offset points',
    arrowprops=dict(arrowstyle='->', color='red', lw=1.5))
ax1.set_xlabel('Corrected mass flow rate  $\dot{m}_{ref}$  [kg/s]', fontsize=12)
ax1.set_ylabel('Stagnation pressure ratio  $p_{02}/p_{01}$', fontsize=12)
ax1.set_title('(a)  Compressor Operation Map', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10, loc='upper right')
ax1.grid(True, alpha=0.3)

# --- (b) Scale vs p01 ---
ax2 = axes[1]
scales_all = [r['scale'] for _, r in results]
p01_all    = [r['p01']/1000 for _, r in results]
colors     = ['green' if r['all_ok'] else 'salmon' for _, r in results]
ax2.scatter(p01_all, scales_all, c=colors, s=18, alpha=0.7)
ax2.axhline(best['scale'], color='red', ls='--', lw=1.5,
            label=f'Max scale = {best["scale"]:.4f}')
ax2.axvline(200, color='gray', ls=':', lw=1, label='$p_{min}$ = 200 kPa')
ax2.set_xlabel('$p_{01}$  [kPa]', fontsize=12)
ax2.set_ylabel('Scale  $l_{model}/l_{prototype}$', fontsize=12)
ax2.set_title('(b)  Scale vs Inlet Pressure', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ME302_results.png', dpi=150, bbox_inches='tight')
plt.savefig('ME302_results.pdf', dpi=150, bbox_inches='tight')
print("Plots saved as ME302_results.png / .pdf")
plt.show()

## 9. Summary Table — Top 10 Operating Points by Scale

In [ ]:
valid_sorted = sorted(valid_results, key=lambda x: x[1]['scale'], reverse=True)

print(f"{'Rank':>4}  {'Idx':>4}  {'Scale':>10}  {'p01 (kPa)':>10}  "
      f"{'mdot_ref':>10}  {'p0_ratio':>10}  {'p4 (kPa)':>10}")
print("-" * 72)
for rank, (idx, r) in enumerate(valid_sorted[:10], 1):
    print(f"{rank:>4}  {idx:>4}  {r['scale']:>10.6f}  {r['p01']/1000:>10.4f}  "
          f"{mdot_ref_data[idx]:>10.4f}  {p0_ratio_data[idx]:>10.6f}  {r['p4']/1000:>10.4f}")